In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    broadcast, col, collect_set, concat_ws, current_timestamp, date_format, datediff,
    expr, floor, length, lit, lpad, months_between, row_number, size, struct, to_json,
    trim, when,try_to_timestamp
)

In [0]:
# ---- tables and paths ------------------------------------------------------
BRONZE_TABLE      = "skypoints.bronze.raw_member"
COUNTRY_REF_TABLE = "skypoints.silver.country_lookup"      # fixed list, loaded by setup_country_lookup_table
COUNTRY_REF_CODE  = "country_code"                         # column in the reference table that holds the code
CLEAN_TABLE       = "skypoints.silver.member"        # interim target, replaced by the country split (design 6)
REJECTS_TABLE     = "skypoints.silver.member_rejects"
CHECKPOINT_PATH   = "/Volumes/skypoints/silver/members/member_checkpoints"

# Idempotent append for the reject table. If the checkpoint is ever reset, bump the suffix
# (v1 -> v2), otherwise Delta ignores the first batches as "already written".
REJECTS_TXN_APP_ID = "silver_member_rejects_v1"

# ---- rules from the design doc (section 5.2) -----------------------------
STALE_DAYS = 90
MAX_LEN = {
    "member_name": 255,
    "member_id":   18,
    "tier_code":   5,
    "agent_name":  255,
    "state":       5,
    "country_code": 5,
    "is_active":   1,
}

# columns of the raw bronze row, kept as JSON in the reject table
BRONZE_COLS = [
    "member_id", "member_name", "country", "dob", "enrollment_date", "last_flight_date",
    "tier", "is_active", "agent_name", "state", "post_code",
    "_rescued_data", "source_file_name", "file_date", "ingestion_timestamp",
]

# column order of silver.member_clean
CLEAN_COLS = [
    "member_name", "member_id", "enrollment_date", "last_flight_date", "tier_code", "agent_name",
    "state", "country_code", "post_code", "dob", "is_active", "age", "stale_member",
    "file_date", "ingestion_timestamp", "source_file_name",
]

In [0]:
%sql
CREATE TABLE IF NOT EXISTS skypoints.silver.member_rejects
(
    member_name         STRING     COMMENT 'Trimmed raw value, can be NULL',
    member_id           STRING     COMMENT 'Trimmed raw value, can be NULL',
    reject_reason       STRING     COMMENT 'One or more codes written as column:CHECK, joined with "; "',
    _rescued_data       STRING     COMMENT 'The full raw bronze row as JSON',
    source_file_name    STRING,
    file_date           DATE,
    ingestion_timestamp TIMESTAMP  COMMENT 'Bronze load time of the rejected row',
    rejected_at         TIMESTAMP  COMMENT 'When silver rejected the row'
)
USING DELTA
COMMENT 'Silver quarantine: member rows that failed a validation rule. Nothing is dropped silently.';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS skypoints.silver.member
(
    member_name         STRING    COMMENT 'Key. One row per member',
    member_id           STRING,
    enrollment_date     DATE,
    last_flight_date    DATE,
    tier_code           STRING,
    agent_name          STRING,
    state               STRING,
    country_code        STRING,
    post_code           INT,
    dob                 DATE,
    is_active           STRING,
    age                 INT       COMMENT 'Derived from dob, relative to file_date',
    stale_member        BOOLEAN   COMMENT 'Derived: more than 90 days between last_flight_date and file_date',
    file_date           DATE      COMMENT 'Feed date of the record that set this row',
    ingestion_timestamp TIMESTAMP,
    source_file_name    STRING
)
USING DELTA
CLUSTER BY (member_name)
COMMENT 'Silver: typed, validated members, latest record wins. Interim target until the country split is built.';

In [0]:
# ---- step 1-2: clean the text and convert to types ---------------------------

def trim_and_blank_to_null(column_name):
    """Trim spaces. A value that is empty after trimming becomes NULL (design 5.2)."""
    trimmed_text = trim(col(column_name))
    return when(trimmed_text == "", lit(None).cast("string")).otherwise(trimmed_text)


def trim_and_rename_bronze_columns(bronze_df):
    """Bronze row (all STRING) -> trimmed text, blanks as NULL, silver column names.
    Dates and post code stay as text (*_text columns) until convert_text_to_types().
    The untouched bronze row is kept as JSON in `bronze_row_json` for the reject table."""
    return bronze_df.select(
        trim_and_blank_to_null("member_name").alias("member_name"),
        trim_and_blank_to_null("member_id").alias("member_id"),
        trim_and_blank_to_null("enrollment_date").alias("enrollment_date_text"),
        trim_and_blank_to_null("last_flight_date").alias("last_flight_date_text"),
        trim_and_blank_to_null("tier").alias("tier_code"),              # bronze name: tier
        trim_and_blank_to_null("agent_name").alias("agent_name"),
        trim_and_blank_to_null("state").alias("state"),
        trim_and_blank_to_null("country").alias("country_code"),        # bronze name: country
        trim_and_blank_to_null("post_code").alias("post_code_text"),
        trim_and_blank_to_null("dob").alias("dob_text"),
        trim_and_blank_to_null("is_active").alias("is_active"),
        "file_date", "ingestion_timestamp", "source_file_name",
        to_json(struct(*BRONZE_COLS), {"ignoreNullFields": "false"}).alias("bronze_row_json"),
    )


def parse_strict_date(text_column, date_format_pattern):
    """Text -> DATE, or NULL when the text is not a real date.
    NULL when the text is not exactly 8 digits, is not a real calendar date, or does not
    give the same text back when formatted again (31 April is rejected, not rolled to 1 May)."""
    parsed_date = try_to_timestamp( col(text_column), lit(date_format_pattern)).cast("date")
    is_8_digits = col(text_column).rlike(r"^\d{8}$")
    return if(is_8_digits, parsed_date)


def convert_text_to_types(trimmed_df, known_country_codes_df):
    """Text -> DATE / INT, and mark known country codes (adds `is_known_country`).
    A value that cannot be converted becomes NULL; validate_members() rejects the row
    when the original text was not blank."""
    return (
        trimmed_df
        # dob arrives as DDMMYYYY but loses its leading zero (3051985 instead of 03051985).
        # Pad only 1-8 digit values, because lpad would cut a longer one.
        .withColumn(
            "dob_text_padded",
            when(col("dob_text").rlike(r"^\d{1,8}$"), lpad("dob_text", 8, "0"))
            .otherwise(col("dob_text")),
        )
        .withColumn("enrollment_date",  parse_strict_date("enrollment_date_text", "yyyyMMdd"))
        .withColumn("last_flight_date", parse_strict_date("last_flight_date_text", "yyyyMMdd"))
        .withColumn("dob",              parse_strict_date("dob_text_padded", "ddMMyyyy"))
        .withColumn(
            "post_code",
            when(col("post_code_text").rlike(r"^\d+$"), col("post_code_text").cast("int"))),
        )
        # exact, case-sensitive match against the fixed country list
        .join(broadcast(known_country_codes_df), "country_code", "left")
    )


# ---- step 4-5: derived columns, latest record, member_id check ---------------

def add_age_and_stale_member(valid_df):
    """Derived columns. They use file_date (the feed date), never current_date(),
    so a rerun gives the same result."""
    return (
        valid_df
        .withColumn("age", floor(months_between("file_date", "dob") / 12).cast("int"))
        .withColumn("stale_member", datediff("file_date", "last_flight_date") > STALE_DAYS)
    )


def keep_latest_record_per_member(valid_df):
    """One row per member_name: the newest by file_date, then ingestion_timestamp.
    Bronze has no row number in the file, so a tie goes to the later last_flight_date,
    then to the bronze row text. That keeps the result the same on every run."""
    newest_first = Window.partitionBy("member_name").orderBy(
        col("file_date").desc(),
        col("ingestion_timestamp").desc(),
        col("last_flight_date").desc_nulls_last(),
        col("bronze_row_json").desc(),
    )
    return (
        valid_df
        .withColumn("row_rank", row_number().over(newest_first))
        .where("row_rank = 1")
        .drop("row_rank")
    )




In [0]:
def validate_members(typed_df):
    """All row-level validation rules in one place (design 5.2 / 5.3).

    Each rule gives a reject code "column:CHECK" when it fails, otherwise NULL.
    The failed codes are joined into `reject_reason`, for example
    'state:TOO_LONG; dob:INVALID_DATE'. A valid row gets the empty string.

    Rules: mandatory value, maximum length, valid date / integer, known country.
    The member_id uniqueness rule needs the latest record and the silver table,
    so it runs later in flag_member_id_conflicts()."""
    rules = []

    # 1. mandatory columns must have a value
    mandatory_columns = ["member_name", "member_id", "country_code",
                         "file_date", "ingestion_timestamp", "source_file_name"]
    for column_name in mandatory_columns:
        rules.append(when(col(column_name).isNull(), lit(f"{column_name}:MISSING")))
    # enrollment_date is judged on its text, because the typed column is also NULL when invalid
    rules.append(when(col("enrollment_date_text").isNull(), lit("enrollment_date:MISSING")))

    # 2. a populated value longer than its maximum length rejects the row
    for column_name, max_length in MAX_LEN.items():
        rules.append(when(length(col(column_name)) > max_length, lit(f"{column_name}:TOO_LONG")))

    # 3. text was filled in but did not convert to a DATE / INT
    conversions = [
        ("enrollment_date",  "enrollment_date_text",  "INVALID_DATE"),
        ("last_flight_date", "last_flight_date_text", "INVALID_DATE"),
        ("dob",              "dob_text",              "INVALID_DATE"),
        ("post_code",        "post_code_text",        "INVALID_INT"),
    ]
    for typed_column, text_column, error_code in conversions:
        rules.append(when(
            col(text_column).isNotNull() & col(typed_column).isNull(),
            lit(f"{typed_column}:{error_code}"),
        ))

    # 4. country code must exist in the reference table
    rules.append(when(
        col("country_code").isNotNull() & col("is_known_country").isNull(),
        lit("country_code:NOT_IN_REFERENCE"),
    ))

    return (
        typed_df
        .withColumn("reject_reason", concat_ws("; ", *rules))
        .drop("is_known_country")
    )

In [0]:
def read_known_country_codes(session):
    """Country codes from the reference table: trimmed, no blanks, no duplicates.
    `is_known_country` is the marker column used by validate_members()."""
    return (
        session.table(COUNTRY_REF_TABLE)
        .select(trim(col(COUNTRY_REF_CODE)).alias("country_code"))
        .where(col("country_code").isNotNull() & (col("country_code") != ""))
        .distinct()
        .withColumn("is_known_country", lit(True))
    )



def build_reject_table_rows(rejected_df):
    """Shape the failed rows like silver.member_rejects.
    `_rescued_data` holds the full raw bronze row as JSON."""
    return rejected_df.select(
        "member_name", "member_id", "reject_reason",
        col("bronze_row_json").alias("_rescued_data"),
        "source_file_name", "file_date", "ingestion_timestamp",
        current_timestamp().alias("rejected_at"),
    )


def append_to_rejects_table(reject_rows_df, batch_id):
    """Append the rejects. txnAppId + txnVersion make Delta skip the write if this batch
    was already written, so a rerun does not create duplicate rejects."""
    (reject_rows_df.write.format("delta").mode("append")
        .option("txnAppId", REJECTS_TXN_APP_ID)
        .option("txnVersion", batch_id)
        .saveAsTable(REJECTS_TABLE))


def upsert_into_member_clean(session, members_df):
    """Insert new members and update existing ones, never with older data.
    The file_date guard stops an old or late file overwriting newer data,
    and running the same batch twice gives the same result.

    NEXT STEP (design section 6): replace this MERGE with the country split."""
    members_df.createOrReplaceTempView("members_to_upsert")
    session.sql(f"""
        MERGE INTO {CLEAN_TABLE} target
        USING members_to_upsert source
        ON target.member_name = source.member_name
        WHEN MATCHED AND (source.file_date > target.file_date
                          OR (source.file_date = target.file_date
                              AND source.ingestion_timestamp >= target.ingestion_timestamp))
            THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)


def log_batch_summary(batch_id, validated_df, failed_validation_df,
                      passed_validation_df):
    """One line per batch with the row counts."""
    rows_in_batch          = validated_df.count()
    rows_failed_validation = failed_validation_df.count()
    rows_passed_validation = passed_validation_df.count()
  

    print(
        f"batch {batch_id}: in={rows_in_batch} | "
        f"rejected={rows_failed_validation} "
        f"(validation={rows_failed_validation}) | "
        f"superseded duplicates={rows_passed_validation} | "
        f"upserted={rows_passed_validation}"
    )

In [0]:
def load_member_batch_to_silver(bronze_batch_df, batch_id):
    """Called once per micro-batch by foreachBatch.
      1. clean the text, convert to types, validate  -> failed rows go to member_rejects
      2. add age / stale_member, keep the latest record per member
      3. check member_id is unique                   -> failed rows go to member_rejects
      4. write the rejects, then upsert the good rows into member_clean
    Every write is safe to rerun (design 6.4)."""
    session = bronze_batch_df.sparkSession

    # 1. clean, convert, validate
    validated_df = (
        bronze_batch_df
        .transform(trim_and_rename_bronze_columns)
        .transform(convert_text_to_types, read_known_country_codes(session))
        .transform(validate_members)
        .persist()
    )
    failed_validation_df = validated_df.where(col("reject_reason") != "")
    passed_validation_df = validated_df.where(col("reject_reason") == "")

    # 2. derived columns, latest record per member
    latest_per_member_df = (
        passed_validation_df
        .transform(add_age_and_stale_member)
        .transform(keep_latest_record_per_member)
    )

    append_to_rejects_table(build_reject_table_rows(failed_validation_df), batch_id)
    upsert_into_member_clean(session, latest_per_member_df)

    log_batch_summary(batch_id, validated_df, failed_validation_df,
                     latest_per_member_df)

    validated_df.unpersist()
    id_checked_df.unpersist()

In [0]:
# Pre-flight: the country reference must exist and hold the code column
ref = spark.table(COUNTRY_REF_TABLE)
assert COUNTRY_REF_CODE in ref.columns, f"{COUNTRY_REF_CODE} not found. Columns are: {ref.columns}"
print("country reference columns:", ref.columns, "| rows:", ref.count())

In [0]:
query = (
    spark.readStream.table(BRONZE_TABLE)
    .writeStream
    .foreachBatch(load_member_batch_to_silver)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)                   # process what is there, then stop
    .start()
)
query.awaitTermination()

In [0]:
%sql
-- Why rows were rejected (a row with several problems appears once, with all its codes)
SELECT reject_reason, count(*) AS rows
FROM skypoints.silver.member_rejects
GROUP BY reject_reason
ORDER BY rows DESC;

In [0]:
%sql
select * from skypoints.silver.member_rejects order by rejected_at desc;

In [0]:
%sql
select * from skypoints.silver.member_clean;